# Activation extraction

This notebook builds the stimulus set and extracts contextual number representations from the models used in the paper.

For a first run, keep `MODELS_TO_RUN = ["bert"]`. BERT is small enough to verify the full pipeline. To reproduce all activation files, change that one list to include GPT-2, Qwen2.5, and Qwen2.5-Math.

The notebook loads **one model at a time**, saves its pickle, then frees memory before loading the next model.

## 0. Small setup

In [5]:
from pathlib import Path
import os, sys, subprocess

REPO_ROOT = '/content/number_geometry_repo'
REPO_ROOT = Path(REPO_ROOT)

sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo root:", REPO_ROOT)

Repo root: /content/number_geometry_repo


## 1. Imports and settings

In [6]:
import gc
import pickle
from collections import defaultdict

import numpy as np
import torch
from tqdm.auto import tqdm

from number_geometry.stimuli import load_all_stimuli
from number_geometry.extraction import EmbeddingExtractor
from number_geometry.models import load_model, MODEL_SPECS

CORPUS_DIR = REPO_ROOT / "corpus_data"
ACTIVATION_DIR = REPO_ROOT / "activations"
ACTIVATION_DIR.mkdir(exist_ok=True)

# Beginner default. Full paper extraction:
# MODELS_TO_RUN = ["bert", "gpt2", "qwen2.5", "qwen2.5-math"]
MODELS_TO_RUN = ["bert"]

print("Available model names:", list(MODEL_SPECS))
print("Models selected:", MODELS_TO_RUN)

Available model names: ['bert', 'gpt2', 'qwen2.5', 'qwen2.5-math']
Models selected: ['bert']


## 2. Build the exact stimulus list

In [7]:
all_data = load_all_stimuli(CORPUS_DIR)
print("Total samples:", len(all_data))
print("First example:", all_data[0])

Total samples: 3690
First example: {'task': 'real_insert', 'subset': 'real_data', 'val': 1, 'format': 'digit', 'text': 'Originally 1 built in Japan and sailed as', 'target_idx': 1, 'target_token': '1'}


## 3. Token-alignment sanity check

In [8]:
def inspect_alignment(dataset, tokenizer, extractor, n=5, seed=0):
    rng = np.random.default_rng(seed)
    for idx in rng.choice(len(dataset), size=min(n, len(dataset)), replace=False):
        item = dataset[int(idx)]
        token_ids, char_span = extractor.target_token_indices(
            item["text"], tokenizer, item["target_idx"], concept=item["target_token"]
        )
        encoded = tokenizer(item["text"], return_tensors="pt")
        decoded = tokenizer.decode(encoded["input_ids"][0, token_ids]).strip() if token_ids else ""
        print("OK" if item["target_token"].lower() in decoded.lower() else "CHECK",
              "| target=", item["target_token"], "| decoded=", repr(decoded), "|", item["text"])

## 4. Extract and save activations

In [9]:
for model_name in MODELS_TO_RUN:
    print("\n" + "=" * 70)
    print("Loading", model_name)
    model, tokenizer = load_model(model_name)
    extractor = EmbeddingExtractor()

    inspect_alignment(all_data, tokenizer, extractor, n=5)

    grouped = defaultdict(list)
    zero_vectors = 0
    for item in tqdm(all_data, desc=f"Extracting {model_name}"):
        emb_layers = extractor.extract_contextual_single(
            text=item["text"],
            model=model,
            tokenizer=tokenizer,
            target_word_index=[item["target_idx"]],
            concept=item["target_token"],
        )
        arr = np.stack(emb_layers)  # [layers, hidden]
        if np.all(arr[-1] == 0):
            zero_vectors += 1
        key = (item["task"], item["subset"], item["val"], item["format"])
        grouped[key].append(arr)

    output = {key: np.stack(values).astype(np.float32) for key, values in grouped.items()}
    path = ACTIVATION_DIR / f"spatial_analysis_{model_name}.pkl"
    with path.open("wb") as f:
        pickle.dump(output, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("Saved:", path)
    print("Zero-vector samples:", zero_vectors)
    first_key = next(iter(output))
    print("Example key / shape:", first_key, output[first_key].shape)

    # Important for Colab / consumer GPUs: release each model before loading the next.
    del output, grouped, extractor, model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Loading bert


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OK | target= two | decoded= 'two' | following two references provide analysis and initial
OK | target= 2 | decoded= '2' | is further divided into 2 complementary clades.
OK | target= one | decoded= 'one' | Metallic badges were often made by one jewelers
OK | target= three | decoded= 'three' | Uc Davis football three team represented the University
OK | target= 8 | decoded= '8' | The number before 9 is 8


Extracting bert:   0%|          | 0/3690 [00:00<?, ?it/s]

Saved: /content/number_geometry_repo/activations/spatial_analysis_bert.pkl
Zero-vector samples: 0
Example key / shape: ('real_insert', 'real_data', 1, 'digit') (100, 12, 768)


## 5. Next step

Open `representation_analysis.ipynb`. It will automatically discover the new `activations/spatial_analysis_*.pkl` files.